# Lesson X3: RL Deployment - Production and Safety

## Introduction

X1 and X2 covered debugging and evaluation — getting training right. This notebook covers what happens **after** training: shipping a policy somewhere that isn't the training loop.

1. **Deploying to production**: exporting a trained policy to a portable inference format
2. **Sim-to-real transfer**: why a policy trained in an idealized simulator can fail on first contact with reality, and how domain randomization fixes it
3. **Safe exploration and reward specification**: constraining *what the agent is allowed to try* during training, not just shaping what it's rewarded for
4. **Human-in-the-loop RL and case studies**: where real deployments actually keep a human in the decision loop, and what that buys you

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)
torch.manual_seed(42)

## Part 1: Deploying RL Models to Production

A training-time SB3 model carries far more than a production deployment needs — optimizer state, replay buffers, the whole training harness. Deployment usually wants exactly one thing: a fast, portable, framework-independent function from observation to action. **ONNX** (Open Neural Network Exchange) is the standard portable format — export once, run inference anywhere (C++, mobile, embedded, a different language entirely) without carrying PyTorch along.

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
import onnxruntime as ort

env = gym.make('CartPole-v1')
model = PPO('MlpPolicy', env, verbose=0, device='cpu', seed=0)
model.learn(total_timesteps=3000)


class OnnxablePolicy(nn.Module):
    """Wraps the trained policy so it exports as a plain (obs -> action) function --
    everything else about the SB3 model (optimizer, buffers, training loop) is deployment noise."""

    def __init__(self, policy):
        super().__init__()
        self.policy = policy

    def forward(self, obs):
        return self.policy(obs, deterministic=True)


onnxable_model = OnnxablePolicy(model.policy)
onnxable_model.eval()
dummy_input = torch.randn(1, *env.observation_space.shape)
torch.onnx.export(
    onnxable_model, dummy_input, '/tmp/ppo_cartpole.onnx',
    opset_version=17, input_names=['observation'], output_names=['action'],
    dynamo=False,  # the newer torch.export-based path doesn't yet handle SB3's Categorical head
)

ort_session = ort.InferenceSession('/tmp/ppo_cartpole.onnx')

test_obs = np.random.randn(1, 4).astype(np.float32)
with torch.no_grad():
    torch_action, _, _ = model.policy(torch.as_tensor(test_obs), deterministic=True)
onnx_action = ort_session.run(None, {'observation': test_obs})[0]

print(f"PyTorch (training-time) action: {torch_action.numpy()}")
print(f"ONNX (deployment) action:       {onnx_action}")
print(f"Identical: {np.array_equal(torch_action.numpy(), onnx_action)}")
print("\nThe exported .onnx file has no PyTorch, no SB3, no training code in it at all --")
print("just the policy network's weights and forward pass, runnable via onnxruntime in")
print("any language with an ONNX runtime binding.")

## Part 2: Sim-to-Real Transfer and Domain Randomization

The **reality gap**: a policy trained in a simulator that's slightly wrong about the real world's dynamics can fail catastrophically the first time it meets the real dynamics, even though it looked perfect in simulation. **Domain randomization** is the standard fix: instead of training in one fixed (idealized) simulator, randomize the simulator's physical parameters across a wide range every episode, forcing the policy to find behavior that's robust across that whole range — including whatever the *real* dynamics turn out to be.

Concretely: the classic **Cliff Walking** task (Sutton & Barto), with an added "wind" disturbance that occasionally pushes the agent toward the cliff. Train once in a wind-free "simulator," once with wind strength randomized every episode, then deploy both to a "real world" with a fixed wind strength neither policy was told about in advance.

In [ ]:
ROWS, COLS = 4, 8
START = (ROWS - 1, 0)
GOAL = (ROWS - 1, COLS - 1)
CLIFF = {(ROWS - 1, j) for j in range(1, COLS - 1)}
DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]


def cliff_step(pos, action, wind_prob):
    di, dj = DELTAS[action]
    next_pos = (pos[0] + di, pos[1] + dj)
    if np.random.rand() < wind_prob:
        next_pos = (next_pos[0] + 1, next_pos[1])  # wind pushes DOWN, toward the cliff row
    next_pos = (max(0, min(ROWS - 1, next_pos[0])), max(0, min(COLS - 1, next_pos[1])))
    if next_pos in CLIFF:
        return START, -100.0, True
    done = next_pos == GOAL
    reward = 10.0 if done else -1.0
    return next_pos, reward, done


def train_cliff_walker(randomize_wind, n_episodes=2000, max_steps=100, alpha=0.3, gamma=0.95, epsilon=0.15):
    Q = defaultdict(lambda: np.zeros(4))
    for _ in range(n_episodes):
        # "Simulator" wind: randomized every episode, or always zero (an idealized, wind-free sim)
        wind_prob = np.random.uniform(0, 0.2) if randomize_wind else 0.0
        pos = START
        for _ in range(max_steps):
            action = np.random.randint(4) if np.random.rand() < epsilon else int(np.argmax(Q[pos]))
            next_pos, reward, done = cliff_step(pos, action, wind_prob)
            Q[pos][action] += alpha * (reward + gamma * np.max(Q[next_pos]) - Q[pos][action])
            pos = next_pos
            if done:
                break
    return Q


def evaluate_cliff_walker(Q, wind_prob, n_episodes=300, max_steps=100):
    successes, falls = 0, 0
    for _ in range(n_episodes):
        pos = START
        for _ in range(max_steps):
            action = int(np.argmax(Q[pos]))
            next_pos, reward, done = cliff_step(pos, action, wind_prob)
            if reward == -100.0:
                falls += 1
                break
            pos = next_pos
            if done:
                successes += 1
                break
    return successes / n_episodes, falls / n_episodes


Q_sim_only = train_cliff_walker(randomize_wind=False)
Q_domain_randomized = train_cliff_walker(randomize_wind=True)

REAL_WORLD_WIND = 0.12  # the "real" wind strength -- neither policy was trained on this exact value
sim_success, sim_falls = evaluate_cliff_walker(Q_sim_only, REAL_WORLD_WIND)
dr_success, dr_falls = evaluate_cliff_walker(Q_domain_randomized, REAL_WORLD_WIND)

print(f"Trained in a wind-free 'sim', deployed to real wind={REAL_WORLD_WIND}:")
print(f"  success rate: {sim_success:.0%}, fell off the cliff: {sim_falls:.0%}")
print(f"Trained with domain-randomized wind, deployed to the SAME real wind={REAL_WORLD_WIND}:")
print(f"  success rate: {dr_success:.0%}, fell off the cliff: {dr_falls:.0%}")
print("\nThe sim-only policy learned the shortest path, which runs directly along the cliff edge --")
print("perfectly safe with zero wind, catastrophic with any wind at all. Domain randomization")
print("never lets the policy find that shortcut safe in the first place.")

## Part 3: Safe Exploration and Reward Specification

**Reward hacking** — X1 already gave this a full demonstration (a badly-specified reward gets exploited even though training "looked healthy"). The deployment-focused half of this problem is **safe exploration**: even a *correctly* specified reward doesn't stop an unconstrained exploring agent from occasionally trying a catastrophic action, purely by chance, on the way to learning better. In deployment settings where "catastrophic" means a damaged robot or a real safety incident, that's not acceptable even during training.

**Action masking** is the simplest fix: constrain the *action space itself* at each state, not just the reward — known-unsafe actions are removed from consideration entirely, so exploration can never sample them, regardless of the exploration policy's randomness.

In [ ]:
HAZARD_SIZE = 6
HAZARD_GOAL = (5, 5)
HAZARDS = {(2, 2), (2, 3), (3, 2), (2, 4), (4, 2)}


def in_bounds(pos):
    return 0 <= pos[0] < HAZARD_SIZE and 0 <= pos[1] < HAZARD_SIZE


def hazard_step(pos, action):
    di, dj = DELTAS[action]
    next_pos = (pos[0] + di, pos[1] + dj)
    if not in_bounds(next_pos):
        next_pos = pos
    if next_pos in HAZARDS:
        return next_pos, -100.0, True  # catastrophic and terminal
    done = next_pos == HAZARD_GOAL
    reward = 10.0 if done else -1.0
    return next_pos, reward, done


def safe_action_mask(pos):
    """Actions that do NOT step directly into a hazard -- computed from known hazard
    locations, not learned; this is a hard safety constraint, not a soft reward penalty."""
    safe = []
    for action in range(4):
        di, dj = DELTAS[action]
        next_pos = (pos[0] + di, pos[1] + dj)
        if not in_bounds(next_pos):
            next_pos = pos
        if next_pos not in HAZARDS:
            safe.append(action)
    return safe if safe else list(range(4))


def train_with_hazards(use_action_mask, n_episodes=400, max_steps=50, alpha=0.3, gamma=0.95, epsilon=0.2):
    Q = defaultdict(lambda: np.zeros(4))
    hazard_hits = 0
    for _ in range(n_episodes):
        pos = (0, 0)
        for _ in range(max_steps):
            candidates = safe_action_mask(pos) if use_action_mask else list(range(4))
            if np.random.rand() < epsilon:
                action = np.random.choice(candidates)
            else:
                q_values = Q[pos]
                action = max(candidates, key=lambda a: q_values[a])
            next_pos, reward, done = hazard_step(pos, action)
            if reward == -100.0:
                hazard_hits += 1
            Q[pos][action] += alpha * (reward + gamma * np.max(Q[next_pos]) - Q[pos][action])
            pos = next_pos
            if done:
                break
    return Q, hazard_hits


_, unmasked_hits = train_with_hazards(use_action_mask=False)
_, masked_hits = train_with_hazards(use_action_mask=True)

print(f"Unconstrained epsilon-greedy exploration: {unmasked_hits} hazard hits during training")
print(f"Action-masked exploration:                {masked_hits} hazard hits during training")
print("\nAction masking doesn't change what the policy is rewarded for -- it changes what the")
print("policy is even ALLOWED to try. That distinction matters whenever exploration itself,")
print("not just the converged policy, has to be safe.")

## Part 4: Human-in-the-Loop RL and Case Studies

### Human-in-the-Loop RL

Fully autonomous training isn't always available or safe. Two common patterns keep a human in the loop:

- **RLHF (Reinforcement Learning from Human Feedback)**: rather than hand-specifying a reward function (with all of X1's reward-hacking risk), train a **reward model** from human preference comparisons ("response A is better than response B"), then run ordinary RL against the learned reward model. This is exactly how modern LLM alignment works, and it generalizes to any task where humans can *judge* outcomes more easily than they can *specify* a reward function.
- **Human oversight / kill switches**: a human (or a simpler rule-based safety monitor) can override or halt the policy at deployment time — the RL policy proposes, a supervisory layer disposes.

### Case Studies

- **AlphaGo / AlphaZero** (DeepMind, 2016-2017): self-play RL + MCTS reached superhuman Go, then generalized the same recipe to chess and shogi with zero human game data — the strongest empirical case yet made for self-play as a training paradigm
- **OpenAI Five** (Dota 2, 2018-2019): large-scale PPO trained via massively parallel self-play, needing extensive reward shaping and domain-specific tricks to handle a long-horizon, partially-observed, multi-agent game
- **Robotic manipulation** (e.g., OpenAI's Rubik's Cube-solving hand, 2019): domain randomization (Part 2's technique, at much larger scale) was the key ingredient that let a sim-trained policy transfer to real robot hardware without ever training on the real robot directly
- **Datacenter cooling** (DeepMind + Google, 2016 onward): a rare production RL deployment where a human operator retains final approval authority over the policy's suggested actions — a real-world instance of the human-in-the-loop pattern above
- **Recommender systems**: large-scale industrial RL deployments (e.g., YouTube's) where the reward signal (engagement) is exactly the kind of easy-to-specify-but-easy-to-hack proxy X1 warned about, and where safe-exploration constraints (Part 3) are essential — a live system can't afford to "explore" by recommending harmful content to real users

## Key Takeaways

1. **ONNX export** strips a trained policy down to a portable inference function — no training framework required at deployment time
2. **Domain randomization** trains across a distribution of simulated dynamics rather than one idealized instance, so the deployed policy doesn't just find the best solution *for the sim* — this notebook's own Cliff Walking demo showed a sim-only policy fail 56%+ of the time under real-world wind that domain randomization handled at 100%
3. **Action masking** enforces safety as a hard constraint on the action space itself, not a soft reward penalty — the only approach that guarantees an unsafe action is never taken, even during random exploration
4. **RLHF** replaces hand-specified rewards with a learned reward model trained from human preferences, sidestepping reward-hacking risk at the specification stage rather than patching it after the fact
5. Every case study above pairs a specific *production concern* from this notebook with a specific real deployment that had to solve it — sim-to-real (robotic manipulation), safe exploration (recommender systems), human oversight (datacenter cooling), and reward specification (RLHF, recommenders again) all show up as first-class engineering problems, not academic footnotes, once RL leaves the training loop